# 🇧🇷 Brazil Tracker — 2026 FIFA World Cup
### The Beautiful Data · Signal & Structure by Krystie Dickson

**Run this notebook before and after each Brazil match to generate charts for Substack.**

Brazil's 2026 group stage:
- ✅ Jun 13 · Brazil 1–1 Morocco · MetLife Stadium, NJ
- 🔜 Jun 19 · Brazil vs Haiti · Lincoln Financial Field, Philadelphia
- 🔜 Jun 24 · Scotland vs Brazil · Hard Rock Stadium, Miami

---

## 1 · Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, os
from pathlib import Path
from datetime import datetime

In [2]:

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT        = Path('..')                          # repo root
PROCESSED   = ROOT / 'data' / 'processed'
STATIC      = ROOT / 'data' / 'static'
OUTPUT_DIR  = ROOT / 'outputs' / 'charts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Beautiful Data colors ──────────────────────────────────────────────────
GREEN  = '#3CAC3B'   # Brazil / CONMEBOL
GOLD   = '#C9A227'   # Trophy / highlight
BLUE   = '#2A398D'   # USA / data
RED    = '#E61D25'   # Canada / danger
DARK   = '#12183A'   # Text
LGRAY  = '#F3F4F6'   # Background alt
WHITE  = '#FFFFFF'

def brazil_style():
    """Apply consistent Beautiful Data styling to matplotlib figures."""
    plt.rcParams.update({
        'figure.facecolor': WHITE,
        'axes.facecolor':   WHITE,
        'axes.spines.top':  False,
        'axes.spines.right':False,
        'axes.spines.left': False,
        'axes.grid':        True,
        'grid.color':       '#EEEEEC',
        'grid.linewidth':   0.8,
        'font.family':      'sans-serif',
        'font.size':        11,
        'axes.titlesize':   13,
        'axes.titleweight': 'bold',
        'axes.labelcolor':  DARK,
        'xtick.color':      '#888888',
        'ytick.color':      '#888888',
    })

brazil_style()
print("✓ Setup complete")
print(f"  Output charts → {OUTPUT_DIR.resolve()}")


✓ Setup complete
  Output charts → C:\Users\kadickson\Documents\GitHub\world-cup-2026-hub\outputs\charts


## 2 · Load live match data

In [3]:
# Load latest match data from pipeline
matches_path = PROCESSED / 'latest_matches.json'

if not matches_path.exists():
    print("ERROR: Run scripts/collect_matches.py first")
else:
    all_matches = pd.read_json(matches_path)
    print(f"✓ Loaded {len(all_matches)} total matches")
    print(f"  Played: {len(all_matches[all_matches.status == 'FINISHED'])}")
    print(f"  Last updated: {all_matches.collected_at.max()[:10]}")
    all_matches.head(3)


✓ Loaded 104 total matches
  Played: 14


TypeError: 'Timestamp' object is not subscriptable

In [ ]:
# Filter Brazil matches only
brazil = all_matches[
    (all_matches.home_team.str.contains('Brazil', na=False)) |
    (all_matches.away_team.str.contains('Brazil', na=False))
].copy()

# Add Brazil-perspective columns
brazil['brazil_home'] = brazil.home_team.str.contains('Brazil', na=False)
brazil['brazil_goals'] = brazil.apply(
    lambda r: r.home_score if r.brazil_home else r.away_score, axis=1)
brazil['opponent_goals'] = brazil.apply(
    lambda r: r.away_score if r.brazil_home else r.home_score, axis=1)
brazil['opponent'] = brazil.apply(
    lambda r: r.away_team if r.brazil_home else r.home_team, axis=1)
brazil['result'] = brazil.apply(lambda r:
    'W' if r.brazil_goals > r.opponent_goals else
    'D' if r.brazil_goals == r.opponent_goals else 'L'
    if pd.notna(r.brazil_goals) else 'TBD', axis=1)
brazil['points'] = brazil.result.map({'W':3,'D':1,'L':0,'TBD':0})
brazil['cum_points'] = brazil.points.cumsum()
brazil['gd'] = (brazil.brazil_goals - brazil.opponent_goals).fillna(0)
brazil['cum_gd'] = brazil.gd.cumsum()

played = brazil[brazil.status == 'FINISHED']
print(f"Brazil matches played: {len(played)}")
print(f"Points: {int(played.points.sum())}  |  GD: {int(played.gd.sum())}")
brazil[['date','opponent','brazil_goals','opponent_goals','result','points']].head(10)


## 3 · 2026 campaign at a glance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Brazil 2026 — Group Stage Tracker", 
             fontsize=15, fontweight='bold', color=DARK, y=1.02)

played = brazil[brazil.status == 'FINISHED'].copy()
upcoming = brazil[brazil.status != 'FINISHED'].copy()

# ── Chart 1: Match results timeline ──────────────────────────────────────
ax = axes[0]
result_colors = {'W': GREEN, 'D': GOLD, 'L': RED, 'TBD': LGRAY}

all_games = pd.concat([played, upcoming]).reset_index(drop=True)
for i, row in all_games.iterrows():
    col = result_colors.get(row.result, LGRAY)
    ax.bar(i, 3, color=col, alpha=0.85, width=0.6, edgecolor=WHITE, linewidth=1.5)
    label = f"{int(row.brazil_goals)}–{int(row.opponent_goals)}"             if pd.notna(row.brazil_goals) else "TBD"
    ax.text(i, 1.5, label, ha='center', va='center',
            fontsize=12, fontweight='bold', color=WHITE)
    ax.text(i, -0.4, row.opponent, ha='center', va='top',
            fontsize=9, color=DARK, rotation=15)

ax.set_xlim(-0.5, len(all_games)-0.5)
ax.set_ylim(-1.2, 3.8)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Results", pad=12)
ax.spines['bottom'].set_visible(False)

patches = [mpatches.Patch(color=GREEN, label='Win'),
           mpatches.Patch(color=GOLD,  label='Draw'),
           mpatches.Patch(color=RED,   label='Loss'),
           mpatches.Patch(color=LGRAY, label='Upcoming')]
ax.legend(handles=patches, loc='upper right', fontsize=8, frameon=False)

# ── Chart 2: Cumulative points ────────────────────────────────────────────
ax = axes[1]
if len(played) > 0:
    x = range(len(played))
    ax.fill_between(x, played.cum_points.values, alpha=0.15, color=GREEN)
    ax.plot(x, played.cum_points.values, color=GREEN, linewidth=2.5, marker='o',
            markersize=8, markerfacecolor=WHITE, markeredgecolor=GREEN, markeredgewidth=2)
    for i, v in enumerate(played.cum_points.values):
        ax.text(i, v+0.15, str(int(v)), ha='center', fontsize=10,
                fontweight='bold', color=GREEN)
    # Max possible points line
    max_pts = [3*(i+1) for i in x]
    ax.plot(x, max_pts, color=LGRAY, linewidth=1.5, linestyle='--', label='Max possible')
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"MD{i+1}" for i in x])
    ax.set_ylabel("Cumulative points")
    ax.set_ylim(0, 9.5)
    ax.legend(fontsize=8, frameon=False)
ax.set_title("Points Earned", pad=12)

# ── Chart 3: Goals scored vs conceded ────────────────────────────────────
ax = axes[2]
if len(played) > 0:
    x = range(len(played))
    bar_w = 0.35
    bars_for = ax.bar([i - bar_w/2 for i in x], played.brazil_goals.fillna(0),
                      bar_w, color=GREEN, label='Brazil goals', alpha=0.85)
    bars_ag  = ax.bar([i + bar_w/2 for i in x], played.opponent_goals.fillna(0),
                      bar_w, color=RED,   label='Opponent goals', alpha=0.7)
    ax.set_xticks(list(x))
    ax.set_xticklabels([row.opponent[:8] for _, row in played.iterrows()],
                       rotation=15, ha='right', fontsize=9)
    ax.set_ylabel("Goals")
    ax.legend(fontsize=8, frameon=False)
ax.set_title("Goals For vs Against", pad=12)

plt.tight_layout()
chart_path = OUTPUT_DIR / 'brazil_group_stage_tracker.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor=WHITE)
plt.show()
print(f"\n✓ Chart saved to {chart_path}")
print("  Use this image in your Substack post after each Brazil match")


## 4 · Historical context — Brazil at every World Cup

In [ ]:
# Load historical data (1930-2022)
hist_path = STATIC / 'WorldCupMatches.csv'

if not hist_path.exists():
    print("Download WorldCupMatches.csv from Kaggle first")
    print("kaggle.com/datasets/abecklas/fifa-world-cup")
else:
    hist = pd.read_csv(hist_path)
    
    # Filter Brazil matches
    bra_hist = hist[
        (hist['Home Team Name'].str.strip() == 'Brazil') |
        (hist['Away Team Name'].str.strip() == 'Brazil')
    ].copy()
    
    # Brazil goals for/against per tournament
    bra_hist['bra_goals'] = bra_hist.apply(lambda r:
        r['Home Team Goals'] if r['Home Team Name'].strip()=='Brazil'
        else r['Away Team Goals'], axis=1)
    bra_hist['opp_goals'] = bra_hist.apply(lambda r:
        r['Away Team Goals'] if r['Home Team Name'].strip()=='Brazil'
        else r['Home Team Goals'], axis=1)
    bra_hist['result'] = bra_hist.apply(lambda r:
        'W' if r.bra_goals > r.opp_goals else
        'D' if r.bra_goals == r.opp_goals else 'L', axis=1)
    
    by_year = bra_hist.groupby('Year').agg(
        matches=('result','count'),
        wins=('result', lambda x: (x=='W').sum()),
        draws=('result', lambda x: (x=='D').sum()),
        losses=('result', lambda x: (x=='L').sum()),
        goals_for=('bra_goals','sum'),
        goals_against=('opp_goals','sum'),
    ).reset_index()
    by_year['gd'] = by_year.goals_for - by_year.goals_against
    by_year['win_pct'] = (by_year.wins / by_year.matches * 100).round(1)
    
    print(f"✓ Historical data loaded: {len(bra_hist)} Brazil matches (1930–2022)")
    print(f"  Tournaments: {by_year.Year.nunique()}")
    by_year.tail(10)


In [ ]:
# ── Brazil historical performance chart ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle("Brazil at the FIFA World Cup — 1930 to 2026", 
             fontsize=15, fontweight='bold', color=DARK)

years = by_year.Year.values

# ── Chart 1: Goals for vs against by tournament ───────────────────────────
ax = axes[0]
bar_w = 0.4
ax.bar(years - bar_w/2, by_year.goals_for,  bar_w, color=GREEN, label='Goals scored', alpha=0.85)
ax.bar(years + bar_w/2, by_year.goals_against, bar_w, color=RED, label='Goals conceded', alpha=0.7)

# Highlight title years
title_years = [1958, 1962, 1970, 1994, 2002]
for ty in title_years:
    ax.axvline(ty, color=GOLD, linewidth=2, alpha=0.6, linestyle='--')
    ax.text(ty, ax.get_ylim()[1]*0.92, '🏆', ha='center', fontsize=10)

# Highlight 7-1
ax.annotate('7–1 vs Germany', xy=(2014, 7), xytext=(2008, 11),
            fontsize=9, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax.set_ylabel("Goals")
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, fontsize=9)
ax.legend(fontsize=9, frameon=False)
ax.set_title("Goals Scored vs Conceded per Tournament", pad=8)

# ── Chart 2: Win % per tournament ─────────────────────────────────────────
ax = axes[1]
colors_bar = [GOLD if y in title_years else GREEN for y in years]
bars = ax.bar(years, by_year.win_pct, color=colors_bar, alpha=0.85, width=0.7)

ax.axhline(by_year.win_pct.mean(), color=BLUE, linestyle='--', 
           linewidth=1.5, label=f"Historical avg: {by_year.win_pct.mean():.0f}%")

# Add 2026 so far
if len(played) > 0:
    bra_2026_winpct = played[played.result=='W'].shape[0] / len(played) * 100
    ax.bar(2026, bra_2026_winpct, color=BLUE, alpha=0.7, width=0.7, label='2026 (so far)')
    ax.text(2026, bra_2026_winpct + 1, '2026', ha='center', fontsize=8, color=BLUE)

ax.set_ylabel("Win %")
ax.set_ylim(0, 105)
ax.set_xticks(list(years) + ([2026] if len(played)>0 else []))
ax.set_xticklabels(list(years) + ([2026] if len(played)>0 else []),
                   rotation=45, fontsize=9)
ax.legend(fontsize=9, frameon=False)
ax.set_title("Win Percentage per Tournament  (🏆 = title year)", pad=8)

plt.tight_layout()
chart_path = OUTPUT_DIR / 'brazil_historical.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor=WHITE)
plt.show()
print(f"\n✓ Historical chart saved to {chart_path}")


## 5 · The quarterfinal curse — Brazil's exit rounds since 2002

In [ ]:
# Load tournament-level data
wc_path = STATIC / 'WorldCups.csv'

if not wc_path.exists():
    print("Download WorldCups.csv from Kaggle first")
else:
    wc = pd.read_csv(wc_path)
    print("WorldCups columns:", wc.columns.tolist())
    wc.tail(8)


In [ ]:
# Manual Brazil exit round data (more reliable than parsing from CSV)
# Round encoding: Group=1, R32=2, R16=3, QF=4, SF=5, Final=6, Winner=7
brazil_exits = pd.DataFrame([
    {'year':1930,'round':'Semifinals','round_num':5,'result':'3rd place'},
    {'year':1934,'round':'Round of 16','round_num':3,'result':'Eliminated'},
    {'year':1938,'round':'Semifinals','round_num':5,'result':'3rd place'},
    {'year':1950,'round':'Final','round_num':6,'result':'Runner-up'},
    {'year':1954,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
    {'year':1958,'round':'Winner','round_num':7,'result':'🏆 Champion'},
    {'year':1962,'round':'Winner','round_num':7,'result':'🏆 Champion'},
    {'year':1966,'round':'Group Stage','round_num':1,'result':'Eliminated'},
    {'year':1970,'round':'Winner','round_num':7,'result':'🏆 Champion'},
    {'year':1974,'round':'Final','round_num':6,'result':'Runner-up'},
    {'year':1978,'round':'Final','round_num':6,'result':'3rd place'},
    {'year':1982,'round':'Round of 16','round_num':3,'result':'Eliminated'},
    {'year':1986,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
    {'year':1990,'round':'Round of 16','round_num':3,'result':'Eliminated'},
    {'year':1994,'round':'Winner','round_num':7,'result':'🏆 Champion'},
    {'year':1998,'round':'Final','round_num':6,'result':'Runner-up'},
    {'year':2002,'round':'Winner','round_num':7,'result':'🏆 Champion'},
    {'year':2006,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
    {'year':2010,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
    {'year':2014,'round':'Semifinals','round_num':5,'result':'4th place (7–1)'},
    {'year':2018,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
    {'year':2022,'round':'Quarterfinal','round_num':4,'result':'Eliminated'},
])

# 2026 format has one extra round (R32), so adjust round numbers
# Winner in 2026 = round 8 instead of 7
brazil_exits_recent = brazil_exits[brazil_exits.year >= 2002].copy()
print("Brazil since last title (2002):")
print(brazil_exits_recent[['year','round','result']].to_string(index=False))
print(f"\nQF exits since 2002: {(brazil_exits_recent.round_num==4).sum()} out of {len(brazil_exits_recent)-1} tournaments")


In [ ]:
# ── The quarterfinal curse visualization ──────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

round_labels = {1:'Group Stage',2:'Round of 32',3:'Round of 16',
                4:'Quarterfinal',5:'Semifinal',6:'Final',7:'Winner 🏆'}
title_years  = [1958, 1962, 1970, 1994, 2002]
qf_curse     = [2006, 2010, 2018, 2022]

years_all  = brazil_exits.year.values
rounds_all = brazil_exits.round_num.values

# Line
ax.plot(years_all, rounds_all, color='#CCCCCC', linewidth=1.5, zorder=1)

# Points — colored by outcome
for _, row in brazil_exits.iterrows():
    if row.year in title_years:
        col, size, zorder = GOLD, 120, 4
    elif row.year in qf_curse:
        col, size, zorder = RED, 100, 4
    elif row.round_num >= 5:
        col, size, zorder = GREEN, 100, 4
    else:
        col, size, zorder = BLUE, 80, 3
    ax.scatter(row.year, row.round_num, color=col, s=size, 
               zorder=zorder, edgecolors=WHITE, linewidths=1.5)

# QF curse shading
ax.axhspan(3.5, 4.5, alpha=0.06, color=RED)
ax.text(2013.5, 4.52, 'Quarterfinal exit zone', fontsize=9, 
        color=RED, alpha=0.7, fontstyle='italic')

# 2026 marker (in progress)
ax.scatter(2026, 1.5, color=BLUE, s=80, zorder=4, 
           marker='D', edgecolors=WHITE, linewidths=1.5)
ax.text(2026, 1.7, '2026
(in progress)', ha='center', fontsize=8, color=BLUE)
ax.annotate('7–1 vs Germany', xy=(2014, 5), xytext=(2009, 5.8),
            fontsize=8, color=RED, fontstyle='italic',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))

ax.set_yticks(list(round_labels.keys()))
ax.set_yticklabels(list(round_labels.values()), fontsize=10)
ax.set_xticks(years_all)
ax.set_xticklabels(years_all, rotation=45, fontsize=9)
ax.set_ylim(0.5, 7.8)
ax.set_title("Brazil's World Cup Journey — Exit Round by Tournament (1930–2026)", 
             fontsize=13, fontweight='bold', color=DARK, pad=12)

patches = [
    mpatches.Patch(color=GOLD,  label='🏆 Title'),
    mpatches.Patch(color=GREEN, label='Semifinal+'),
    mpatches.Patch(color=RED,   label='Quarterfinal exit (the curse)'),
    mpatches.Patch(color=BLUE,  label='Earlier exit / In progress'),
]
ax.legend(handles=patches, loc='lower left', fontsize=9, frameon=True,
          framealpha=0.9, edgecolor='#EEEEEE')

plt.tight_layout()
chart_path = OUTPUT_DIR / 'brazil_exit_rounds.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor=WHITE)
plt.show()
print(f"\n✓ Quarterfinal curse chart saved to {chart_path}")
print("  This is your hero chart for the Brazil series on Substack")


## 6 · Run after each Brazil match

**After Brazil vs Haiti (Jun 19) and Scotland vs Brazil (Jun 24), run this section to regenerate all charts with fresh data from the pipeline.**

In [ ]:
# ── Quick match summary — run after any Brazil game ──────────────────────
played = brazil[brazil.status == 'FINISHED'].sort_values('date')

print("=" * 50)
print("BRAZIL 2026 — MATCH SUMMARY")
print("=" * 50)
for _, row in played.iterrows():
    home_away = "vs" if row.brazil_home else "@"
    print(f"  {row.date}  {home_away} {row.opponent:<20} "
          f"{int(row.brazil_goals)}–{int(row.opponent_goals)}  "
          f"{'  ' + row.result}  ({int(row.points)} pts)")

print("-" * 50)
print(f"  Total: {int(played.points.sum())} pts  |  "
      f"GD: {int(played.gd.sum()):+d}  |  "
      f"Goals: {int(played.brazil_goals.sum())} for, "
      f"{int(played.opponent_goals.sum())} against")
print("=" * 50)

# Group C standings snapshot
print("\nGroup C standings (from pipeline):")
try:
    standings = pd.read_json(PROCESSED / 'latest_standings.json')
    group_c = standings[standings.group.str.contains('C', na=False)]
    print(group_c[['team','played','won','drawn','lost',
                   'goals_for','goals_against','points']]
          .sort_values('points', ascending=False)
          .to_string(index=False))
except:
    print("  Run update_scoreboard.py to get group standings")


## 7 · Charts ready for Substack

After running all cells, your charts are saved to `outputs/charts/`:

| File | Use for |
|------|---------|
| `brazil_group_stage_tracker.png` | Post after each Brazil match |
| `brazil_historical.png` | Brazil series deep dive post |
| `brazil_exit_rounds.png` | The quarterfinal curse story |

**Publishing tip:** The exit rounds chart is your best standalone visual — it tells the entire Brazil story in one image. Use it as the cover image for your Brazil series intro post.
